In [1]:
import sys
import os
import pandas as pd

# 1. Tell notebook to look "up" one level for the 'src' folder
sys.path.append(os.path.abspath('..'))
from src.tokenizer import clean_text

# --- STEP 1: Process News (Combine v1 and v2) ---
news_path_1 = '../data/Sarcasm_Headlines_Dataset.json'
news_path_2 = '../data/Sarcasm_Headlines_Dataset_v2.json'

# Load both JSON files
news_df_1 = pd.read_json(news_path_1, lines=True)
news_df_2 = pd.read_json(news_path_2, lines=True)

# Combine them into one big News DataFrame
news_df = pd.concat([news_df_1, news_df_2], ignore_index=True)

# Drop any duplicate headlines (v1 and v2 share some of the same articles)
news_df = news_df.drop_duplicates(subset=['headline'])

# Standardize the columns
news_df = news_df[['headline', 'is_sarcastic']].rename(
    columns={'headline': 'text', 'is_sarcastic': 'label'}
)

# Process Reddit (SARC) ---
reddit_path = '../data/train-balanced-sarcasm.csv'
reddit_df = pd.read_csv(reddit_path)

# Standardize the columns
reddit_df = reddit_df[['comment', 'label']].rename(
    columns={'comment': 'text'}
)

# Sample 50,000 from Reddit 
reddit_df = reddit_df.sample(n=min(50000, len(reddit_df)), random_state=42)


combined_df = pd.concat([news_df, reddit_df], ignore_index=True)
combined_df = combined_df.dropna(subset=['text'])

# Clean the text using your tokenizer
print(" Cleaning text data... (this might take a few seconds)")
combined_df['text'] = combined_df['text'].apply(clean_text)

# Save the final processed dataset
os.makedirs('../data', exist_ok=True)
combined_df.to_csv('../data/processed_sarcasm.csv', index=False)
print(f" Combined Dataset Created! Total rows: {len(combined_df)}")

 Cleaning text data... (this might take a few seconds)


 Combined Dataset Created! Total rows: 78503
